# SIFE-LDM V2 — NLP & Code Generation Pivot

**Architecture:** Unified SIFE Transformer (Complex-valued) with Phase-Coherent Diffusion.

**Dataset:** WikiText-103, CodeAlpaca, MBPP.

**Hardware:** Automatically scales for **TPU v2-8**, **L4**, or **A100** GPUs.

**Goal:** Generate coherent text and programming logic using physics-guided latent fields.

## Cell 1: Workspace Setup & Environment
Detects whether you're using the Antigravity Extension, Google Drive, or a direct upload, and sets up the working directory accordingly.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# --- 1. Install Missing Dependencies ---
print('📦  Installing/Verifying required libraries...')
os.system('pip install -q flax optax datasets')

# --- 2. Workspace Detection & Sync ---
WORKING_DIR = '/content/sife-ldm'
GITHUB_REPO = 'https://github.com/vicekha/SIFE-LDM-.git'

if os.path.exists('/content/sife') and os.path.exists('/content/scripts'):
    WORKING_DIR = '/content'
else:
    if not os.path.exists(WORKING_DIR):
        os.system(f'git clone {GITHUB_REPO} {WORKING_DIR}')
    os.chdir(WORKING_DIR)
    os.system('git pull origin main')

os.environ['PYTHONPATH'] = os.getcwd()

# Verify critical files
critical = ['sife/model.py', 'train.py', 'scripts/quick_start_data.py']
missing = [f for f in critical if not os.path.exists(f)]
if missing:
    print(f'❌  MISSING FILES: {missing}')
else:
    print('✅  Environment ready for SIFE-NLP training.')

## Cell 2: TPU Initialization & Auto-Scaling
Detects your Colab TPU and automatically configures optimal training parameters.

In [ ]:
import os, sys, json, jax, jax.numpy as jnp

# Prevent JAX from pre-allocating all memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

print(f'Python: {sys.version.split()[0]}')
print(f'JAX:    {jax.__version__}')

def setup_accelerator_and_scale():
    """Detect TPU/GPU type and return optimal training params."""
    try:
        import jax.tools.colab_tpu
        jax.tools.colab_tpu.setup_tpu()
    except (KeyError, ImportError):
        pass
        
    devices = jax.devices()
    platform = devices[0].platform
    num_cores = len(devices)
    
    if platform == 'tpu':
        tpu_name = str(devices[0])
        print(f'\n🖥️  TPU Detected: {tpu_name} ({num_cores} cores)')
        if num_cores >= 8:
            cfg = {'batch_size': 128, 'embed_dim': 256, 'base_features': 128,
                   'max_steps': 100000, 'accel_name': 'TPU v2-8', 'num_cores': num_cores}
        else:
            cfg = {'batch_size': 64, 'embed_dim': 192, 'base_features': 96,
                   'max_steps': 75000, 'accel_name': 'TPU', 'num_cores': num_cores}
    
    elif platform == 'gpu':
        gpu_kind = devices[0].device_kind
        print(f'\n🖥️  GPU Detected: {gpu_kind} ({num_cores} device)')
        
        if 'A100' in gpu_kind:
            # A100 40GB/80GB can handle much larger embed dims and batch sizes
            cfg = {'batch_size': 128, 'embed_dim': 512, 'base_features': 128,
                   'max_steps': 100000, 'accel_name': 'A100 GPU', 'num_cores': num_cores}
            print('🚀  NVIDIA A100 Detected — Enabling ULTRA-PERFORMANCE configuration')
        elif 'V100' in gpu_kind:
            cfg = {'batch_size': 96, 'embed_dim': 256, 'base_features': 128,
                   'max_steps': 80000, 'accel_name': 'V100 GPU', 'num_cores': num_cores}
            print('⚡  NVIDIA V100 Detected — Enabling HIGH-PERFORMANCE configuration')
        elif 'L4' in gpu_kind:
            cfg = {'batch_size': 64, 'embed_dim': 192, 'base_features': 96,
                   'max_steps': 75000, 'accel_name': 'L4 GPU', 'num_cores': num_cores}
            print('⚡  NVIDIA L4 Detected — Enabling MEDIUM configuration')
        else:
            cfg = {'batch_size': 32, 'embed_dim': 128, 'base_features': 64,
                   'max_steps': 50000, 'accel_name': 'T4 GPU', 'num_cores': num_cores}
            print(f'💡  Standard GPU ({gpu_kind}) Detected — Enabling BASE configuration')
            
    else:
        print('\n⚠️  No Accelerator detected! Using CPU.')
        cfg = {'batch_size': 4, 'embed_dim': 64, 'base_features': 32,
               'max_steps': 10000, 'accel_name': 'CPU', 'num_cores': 0}
               
    print(f'\n📋  Optimization Profile:')
    for k, v in cfg.items():
        print(f'    {k:15s} = {v}')
    return cfg

ACCEL_CONFIG = setup_accelerator_and_scale()
with open('./accel_config.json', 'w') as f:
    json.dump(ACCEL_CONFIG, f, indent=2)
print(f'\n✅  Optimization profile saved.')

## Cell 3: CIFAR-100 Data Download
Downloads and prepares the CIFAR-100 dataset.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

print('📥  Downloading NLP and Code datasets (WikiText-103, CodeAlpaca)...')
os.system('python scripts/quick_start_data.py --mode both')

data_path = './datasets/nlp/nlp_combined_train.txt'
if os.path.exists(data_path):
    print(f'✅  Data ready at: {data_path}')
    with open(data_path, 'r', encoding='utf-8') as f:
        preview = [next(f) for _ in range(5)]
        print('\nPreview:\n' + ''.join(preview))
else:
    print('❌  Download failed.')

## Cell 4: TPU-Optimized V2 Training

Training parameters are **automatically scaled** based on the TPU config detected in Cell 2.

| TPU Tier  | Batch | Embed Dim | Base Feat | Max Steps |
|-----------|-------|-----------|-----------|-----------|
| TPU v2-8  | 128   | 256       | 128       | 100,000   |
| TPU 4-core| 64    | 192       | 96        | 75,000    |
| Smaller   | 32    | 128       | 64        | 50,000    |

**V2 Enhancements:**
- `ImageEncoder` — learned RGB→complex projection
- `LabelEncoder` — CIFAR-100 class labels as complex context
- 15% CFG dropout → class-conditional generation at inference
- `predict_meta_physics()` — dynamic Hamiltonian params
- `PhasePool/PhaseUnpool` — 50% FLOPs reduction

*(Note: We use `train_vision_gpu.py` because the underlying JAX code is hardware-agnostic and automatically utilizes the initialized TPU.)*

In [ ]:
import os, json
os.environ['PYTHONPATH'] = '.'

with open('./accel_config.json', 'r') as f:
    accel_cfg = json.load(f)

print(f"🚀  Training NLP-Transformer on {accel_cfg['accel_name']}...")
print(f"    Batch Size: {accel_cfg['batch_size']}")

cmd = (f"python train.py --data ./datasets/nlp/nlp_combined_train.txt "
       f"--embed_dim {accel_cfg['embed_dim']} --batch_size {accel_cfg['batch_size']} "
       f"--max_steps {accel_cfg['max_steps']} --max_seq_len 1024")

os.system(cmd)

## Cell 5: Monitor Training Loss

Expected loss curve:
- Step 0: ~6.0 (random noise)
- Step 1,000: ~2.0–3.5
- Step 10,000: ~1.2–1.8
- Step 50,000: ~0.8–1.2 (good generative quality)

In [ ]:
import matplotlib.pyplot as plt
import re, os

log_file = 'training_log.txt'
if os.path.exists(log_file):
    steps, losses = [], []
    with open(log_file) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss\s*=\s*([\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

    plt.figure(figsize=(12, 4))
    plt.plot(steps, losses, 'b-', linewidth=1.5, label='Diffusion Loss')
    plt.axhline(y=2.0, color='g', linestyle='--', alpha=0.5, label='Good quality threshold')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('SIFE-LDM V2 Training Curve (CIFAR-100)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('No log file found. Training output is inline above.')

## Cell 6: SIFE-Transformer Inference
Generates text using the physics-guided `inference.py` script.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

PROMPT = 'Biological intelligence is rooted in'
print(f'Generating from prompt: "{PROMPT}"')

os.system(f'python inference.py --checkpoint checkpoints/checkpoint_final --prompt "{PROMPT}" --num_steps 50')